# Stock Market Data Analysis
**Intermediate project | Python, Pandas, Matplotlib, yfinance, Scikit-learn**

This notebook downloads public historical stock data, explores returns and risk, builds technical indicators, and evaluates a next-session closing-price model against a naive baseline.

> Educational analysis only. It is not investment advice. Forecasts are uncertain and should not be used alone for trading decisions.

## 1. Install dependencies

From the project root, run `pip install -r requirements.txt` in your activated virtual environment. Then launch Jupyter with `jupyter notebook`.

In [2]:
from datetime import date, timedelta
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
sns.set_theme()
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
print("Project root:", ROOT.resolve())

ModuleNotFoundError: No module named 'yfinance'

## 2. Download the public dataset

Choose a ticker and date range. Yahoo Finance's end date is exclusive, so the code adds one day to include the selected final date. If your internet or Yahoo Finance access is unavailable, download the historical CSV manually from [Yahoo Finance](https://finance.yahoo.com/) and load it with `pd.read_csv(..., parse_dates=["Date"], index_col="Date")`.

In [ ]:
TICKER = "AAPL"  # Examples: MSFT, NVDA, RELIANCE.NS, TCS.NS, ^NSEI
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=365 * 5)

raw = yf.download(
    TICKER,
    start=START_DATE.isoformat(),
    end=(END_DATE + timedelta(days=1)).isoformat(),
    auto_adjust=False,
    progress=False,
    threads=False,
)
if raw.empty:
    raise RuntimeError("No data returned. Check ticker, dates, internet connection, or download a CSV manually.")
# Recent yfinance versions may return multi-level columns for one ticker.
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
raw = raw.loc[:, ~raw.columns.duplicated()].copy()
raw.index = pd.to_datetime(raw.index)
raw.index.name = "Date"
raw = raw.sort_index().dropna(subset=["Close"])
for col in raw.columns:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

csv_path = DATA_DIR / f"{TICKER.lower().replace('^', '')}_historical.csv"
raw.to_csv(csv_path)
print(f"Downloaded {len(raw):,} rows for {TICKER}")
print("Saved:", csv_path)
display(raw.head())
display(raw.tail())

## 3. Data quality checks

In [ ]:
print("Shape:", raw.shape)
print("\\nColumns:", raw.columns.tolist())
print("\\nMissing values:")
display(raw.isna().sum().to_frame("missing_count"))
print("\\nDuplicate dates:", raw.index.duplicated().sum())
print("\\nDate range:", raw.index.min().date(), "to", raw.index.max().date())
display(raw.describe().T)

## 4. Feature engineering

We calculate daily percentage returns, 20- and 50-session moving averages, 20-session annualized volatility, and drawdown from the running peak.

In [ ]:
df = raw.copy()
df["Daily_Return_%"] = df["Close"].pct_change() * 100
df["MA_20"] = df["Close"].rolling(20).mean()
df["MA_50"] = df["Close"].rolling(50).mean()
df["Volatility_20D_%"] = df["Daily_Return_%"].rolling(20).std() * np.sqrt(252)
df["Rolling_Max"] = df["Close"].cummax()
df["Drawdown_%"] = (df["Close"] / df["Rolling_Max"] - 1) * 100
display(df.tail())

## 5. Visual exploratory data analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
axes[0].plot(df.index, df["Close"], label="Close")
axes[0].plot(df.index, df["MA_20"], label="20-session MA")
axes[0].plot(df.index, df["MA_50"], label="50-session MA")
axes[0].set_title(f"{TICKER}: close price and moving averages")
axes[0].set_ylabel("Price")
axes[0].legend()
axes[1].bar(df.index, df["Volume"], width=1)
axes[1].set_title("Trading volume")
axes[1].set_ylabel("Volume")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["Daily_Return_%"].dropna(), bins=50, kde=True, ax=axes[0])
axes[0].set_title("Daily return distribution")
sns.lineplot(data=df, x=df.index, y="Volatility_20D_%", ax=axes[1])
axes[1].set_title("20-session annualized volatility")
plt.tight_layout()
plt.show()

df["Drawdown_%"].plot(figsize=(14, 4), title="Drawdown from running price peak")
plt.ylabel("Drawdown (%)")
plt.show()

## 6. Summary statistics and risk measures

Annualized volatility is the standard deviation of daily returns multiplied by the square root of 252 trading sessions. Maximum drawdown summarizes the largest peak-to-trough decline in the selected period.

In [ ]:
returns = df["Close"].pct_change().dropna()
summary = pd.Series({
    "First close": df["Close"].iloc[0],
    "Latest close": df["Close"].iloc[-1],
    "Period return (%)": (df["Close"].iloc[-1] / df["Close"].iloc[0] - 1) * 100,
    "Mean daily return (%)": returns.mean() * 100,
    "Daily return standard deviation (%)": returns.std() * 100,
    "Annualized volatility (%)": returns.std() * np.sqrt(252) * 100,
    "Maximum drawdown (%)": df["Drawdown_%"].min(),
    "Mean daily volume": df["Volume"].mean(),
})
display(summary.to_frame("value").round(4))

## 7. Train a next-session closing-price model

For each date *t*, the features use information available on or before *t*, while the target is the close at the next observed trading session (*t+1*). The data is split chronologically: first 80% for training, last 20% for testing. The baseline assumes the next close equals today's close.

This setup avoids random shuffling for time series. It is still a simple educational baseline; it does not include transaction costs, corporate-action modeling, or walk-forward validation.

In [ ]:
model_data = df.copy()
model_data["Target_Next_Close"] = model_data["Close"].shift(-1)
model_data["Close_Lag_1"] = model_data["Close"]
model_data["Close_Lag_2"] = model_data["Close"].shift(1)
model_data["Close_Lag_5"] = model_data["Close"].shift(4)
model_data["Return_Lag_1"] = model_data["Close"].pct_change()
model_data["MA_5"] = model_data["Close"].rolling(5).mean()
model_data["MA_20_model"] = model_data["Close"].rolling(20).mean()
model_data["Volume_Change"] = model_data["Volume"].pct_change()

features = [
    "Close_Lag_1", "Close_Lag_2", "Close_Lag_5", "Return_Lag_1",
    "MA_5", "MA_20_model", "Volume_Change"
]
model_data = model_data.replace([np.inf, -np.inf], np.nan)
model_data = model_data.dropna(subset=features + ["Target_Next_Close"])
if len(model_data) < 80:
    raise ValueError("Not enough usable rows. Choose a longer historical period.")

split_idx = int(len(model_data) * 0.8)
train = model_data.iloc[:split_idx]
test = model_data.iloc[split_idx:]
X_train, y_train = train[features], train["Target_Next_Close"]
X_test, y_test = test[features], test["Target_Next_Close"]

model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(
        n_estimators=250, max_depth=8, min_samples_leaf=3,
        random_state=42, n_jobs=-1
    )
)
model.fit(X_train, y_train)
rf_pred = model.predict(X_test)
baseline_pred = X_test["Close_Lag_1"].to_numpy()

print(f"Train rows: {len(train):,} | Test rows: {len(test):,}")


## 8. Evaluate the model against the baseline

MAE and RMSE are measured in price units. Lower is better. R² can be negative when predictions perform worse than predicting the test-set mean. Do not assume that a model is useful simply because it runs successfully.

In [ ]:
def regression_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
    }

metrics = pd.DataFrame({
    "Random Forest": regression_metrics(y_test, rf_pred),
    "Naive baseline": regression_metrics(y_test, baseline_pred),
}).T
display(metrics.round(4))

predictions = pd.DataFrame({
    "Actual_Next_Close": y_test.to_numpy(),
    "RandomForest_Prediction": rf_pred,
    "Naive_Baseline_Today_Close": baseline_pred,
}, index=test.index)
predictions.index.name = "Feature_Date"
display(predictions.head())
predictions.to_csv(OUTPUT_DIR / f"{TICKER.lower().replace('^', '')}_test_predictions.csv")
print("Saved model predictions to outputs/")

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(predictions.index, predictions["Actual_Next_Close"], label="Actual next close")
plt.plot(predictions.index, predictions["RandomForest_Prediction"], label="Random Forest")
plt.plot(predictions.index, predictions["Naive_Baseline_Today_Close"], label="Naive baseline", alpha=0.8)
plt.title(f"{TICKER}: holdout predictions vs actual")
plt.xlabel("Feature date (target is next trading session)")
plt.ylabel("Closing price")
plt.legend()
plt.tight_layout()
plt.show()

mae_rf = metrics.loc["Random Forest", "MAE"]
mae_base = metrics.loc["Naive baseline", "MAE"]
print("Random Forest beats baseline on MAE:", bool(mae_rf < mae_base))

## 9. Save the analyzed data and export a report CSV

In [ ]:
analyzed_path = OUTPUT_DIR / f"{TICKER.lower().replace('^', '')}_analyzed_data.csv"
df.to_csv(analyzed_path)
print("Saved:", analyzed_path)
display(df.tail(10))

## 10. Findings and limitations

Write your own observations after running the notebook:
- What was the total return over the selected period?
- When did volatility increase?
- What was the maximum drawdown?
- Did Random Forest outperform the naive baseline on MAE and RMSE?
- What additional validation would be needed before using a model in a real financial setting?

**Limitations:** historical market data can be revised or missing; the model is not a trading strategy; the chosen features are basic; one holdout split can be sensitive to market regime; transaction costs, slippage, taxes, liquidity, and risk management are not modeled.